In [76]:
import pandas as pd

In [77]:
df = pd.read_csv("./WA_Fn-UseC_-Telco-Customer-Churn.csv")

# Cleaning data:

# Clean col names
cleaned_col_names = []
for c in df.columns:
    c = c[0].lower() + c[1:]
    cleaned_col_names.append(c)
df.columns = cleaned_col_names
df.head()

df_original = df.copy()
# Clean bool data
boolean_cols_to_clean = [
    "partner",
    "dependents",
    "phoneService",
    "onlineSecurity",
    "deviceProtection",
    "techSupport",
    "streamingTV",
    "streamingMovies",
    "paperlessBilling",
    "churn",
    "multipleLines",
    "onlineBackup",
]


for col in boolean_cols_to_clean:
    new_vals = []
    for res in df[col].values:
        if res.lower() == "yes":
            new_vals.append(1)
        elif res.lower() == "no" or res.lower() in {"no phone service", "no internet service"}:
            new_vals.append(0)
        else:
            new_vals.append(res.lower())
    df[col] = new_vals
    # print(set(new_vals))
# df.dtypes
# df.internetService.unique()
# df.onlineSecurity.unique()
# df.onlineSecurity

# Handle category data
categories_col = ["gender", "internetService", "contract", "paymentMethod"]


def parse_str_to_float(val: str):
    try:
        return float(val)
    except:
        return 0


df["totalCharges"] = [parse_str_to_float(val) for val in df.totalCharges.values]


for cat in categories_col:
    df[cat] = [word.lower().replace(" ", "_").replace("-", "_") for word in df[cat].values]


new_col = {}
for cat in categories_col:
    unique_values = set(df[cat].values)
    for unique_value in unique_values:
        col_name = f"{cat}_{unique_value}"
        arr = []
        for value in df[cat].values:
            if value == unique_value:
                arr.append(1)
            else:
                arr.append(0)
        new_col[col_name] = arr

df = pd.concat([df, pd.DataFrame(new_col, index=df.index)], axis=1)
# df = df.drop(columns=["c
# ustomerID", "gender", "internetService", "contract", "paymentMethod"])
# for c in df.columns:
#     print(c, list(df[c].unique()))
# new_col
df.head()
df.dtypes


customerID                                     str
gender                                         str
seniorCitizen                                int64
partner                                      int64
dependents                                   int64
tenure                                       int64
phoneService                                 int64
multipleLines                                int64
internetService                                str
onlineSecurity                               int64
onlineBackup                                 int64
deviceProtection                             int64
techSupport                                  int64
streamingTV                                  int64
streamingMovies                              int64
contract                                       str
paperlessBilling                             int64
paymentMethod                                  str
monthlyCharges                             float64
totalCharges                   

In [78]:
from sklearn.model_selection import train_test_split

df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(df_full_train, test_size=0.25)

df_full_train = df_full_train.reset_index(drop=True)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)

y_train = df_train.churn.values
y_val = df_val.churn.values
y_test = df_test.churn.values

df_train.drop(columns=["churn"])
df_val.drop(columns=["churn"])
df_test.drop(columns=["churn"])


,customerID,gender,seniorCitizen,partner,dependents,tenure,phoneService,multipleLines,internetService,onlineSecurity,...,internetService_no,internetService_fiber_optic,internetService_dsl,contract_two_year,contract_month_to_month,contract_one_year,paymentMethod_mailed_check,paymentMethod_bank_transfer_(automatic),paymentMethod_credit_card_(automatic),paymentMethod_electronic_check
3381,8879-ZKJOF,female,0,0,0,41,1,0,dsl,1,...,0,0,1,0,0,1,0,1,0,0
6180,0201-MIBOL,female,1,0,0,66,1,1,fiber_optic,1,...,0,1,0,1,0,0,0,1,0,0
4829,1600-DILPE,female,0,0,0,12,1,0,dsl,0,...,0,0,1,0,1,0,0,1,0,0
3737,8601-QACRS,female,0,0,0,5,1,1,dsl,0,...,0,0,1,0,1,0,1,0,0,0
4249,7919-ZODZZ,female,0,1,1,10,1,0,dsl,0,...,0,0,1,0,0,1,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2563,5130-IEKQT,male,1,0,0,25,1,1,fiber_optic,0,...,0,1,0,0,1,0,1,0,0,0
2028,4452-ROHMO,female,0,0,0,15,1,0,no,0,...,1,0,0,1,0,0,1,0,0,0
2899,6164-HAQTX,male,0,0,0,71,0,0,dsl,1,...,0,0,1,1,0,0,0,1,0,0
3474,3982-DQLUS,male,1,1,1,65,1,1,fiber_optic,1,...,0,1,0,0,1,0,0,0,0,1


In [79]:
for c in categories_col:
    global_churn = df_full_train.churn.mean()
    df_group = df_full_train.groupby(c).churn.agg(["mean", "count"])
    df_group["diff"] = df_group["mean"] - global_churn
    df_group["risk"] = df_group["mean"] / global_churn
    display(df_group)


# categories_col

,mean,count,diff,risk
gender,,,,
female,0.276824,2796,0.006856,1.025396
male,0.263214,2838,-0.006755,0.974980


,mean,count,diff,risk
internetService,,,,
dsl,0.192347,1934,-0.077621,0.712482
fiber_optic,0.425171,2479,0.155203,1.574895
no,0.077805,1221,-0.192163,0.288201


,mean,count,diff,risk
contract,,,,
month_to_month,0.431701,3104,0.161733,1.599082
one_year,0.120573,1186,-0.149395,0.446621
two_year,0.028274,1344,-0.241694,0.104730


,mean,count,diff,risk
paymentMethod,,,,
bank_transfer_(automatic),0.168171,1219,-0.101797,0.622928
credit_card_(automatic),0.164339,1217,-0.105630,0.608733
electronic_check,0.455890,1893,0.185922,1.688682
mailed_check,0.193870,1305,-0.076098,0.718121


In [ ]:
from sklearn.metrics import mutual_info_score


def mutual_info_churn(series):
    return mutual_info_score(series, df_full_train.churn)


mi = df_full_train[categories_col].apply(mutual_info_churn)
mi.sort_values(ascending=False)

numerical = df_original.select_dtypes(include="number").columns.to_list()

df_full_train[numerical].corrwith(df_full_train.churn)


seniorCitizen     0.141966
tenure           -0.351885
monthlyCharges    0.196805
dtype: float64